In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../src")
import numpy as np
from energy import load_framework, load_force_field, pairwise_lj_params
from widom import widom, rho_framework, henry_coefficient


In [2]:
positions, symbols, L, masses = load_framework("../data/IRMOF-1.cif")
lj_params = load_force_field("../data/force_field/generic_dreiding_uff_mixing_rules.def")
eps_pair, sigma_pair = pairwise_lj_params(symbols, lj_params, guest=(158.5, 3.72))
rho_fw = rho_framework(masses, L)
print(f"rho_fw = {rho_fw:.1f} kg/m^3 (literatuur ca. 590)")


rho_fw = 593.3 kg/m^3 (literatuur ca. 590)


/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/.venv/lib/python3.14/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


In [3]:
# Widom: N inserties, fout via 10 blokgemiddelden. Fout zakt met sqrt(N); 10.000 inserties ~ 0,4 s.
N = 100_000
T = 298.0  # K
rng = np.random.default_rng(seed=100)

boltzmann_mean, sem = widom(N, T, positions, eps_pair, sigma_pair, L, rng, n_blocks=10)
K_H = henry_coefficient(boltzmann_mean, T, rho_fw)
K_H_err = henry_coefficient(sem, T, rho_fw)  # zelfde noemer
print(f"<exp(-U/T)> = {boltzmann_mean:.3f} ± {sem:.3f}")
print(f"K_H(298 K) = {K_H:.3e} ± {K_H_err:.1e} mol/kg/Pa")


<exp(-U/T)> = 5.634 ± 0.027
K_H(298 K) = 3.832e-06 ± 1.8e-08 mol/kg/Pa


## Vergelijking met literatuur

TODO (Kasper): gepubliceerde K_H voor CH4 in IRMOF-1 met DREIDING/UFF bij 298 K opzoeken
(Dubbeldam et al., Angew. Chem. 2007; RASPA-voorbeelden) en hier met bron naast de eigen waarde zetten.
